# E-Commerce Data Pipeline & Analytics Dashboard

This notebook is the Kaggle-friendly analysis companion for the portfolio project. It shows how raw e-commerce data is transformed into a SQLite star schema, then analyzed with SQL for revenue, RFM, product, pricing, return-risk, and city-tier insights.

## 1. Project Context

A mid-size e-commerce business has transactions, product metadata, and customer attributes spread across multiple sources. The goal is to build a repeatable ETL pipeline and answer business questions such as churn risk, product profitability, suspicious pricing, and city-tier repeat behavior.

In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = PROJECT_ROOT / "data" / "database" / "ecommerce.db"
SQL_DIR = PROJECT_ROOT / "sql"

conn = sqlite3.connect(DB_PATH)
print(f"Connected to: {DB_PATH}")

## 2. Warehouse Tables

The ETL loads a star schema with dimensions for date, customer, and product, plus facts for orders and returns.

In [ ]:
tables = pd.read_sql_query("""
SELECT name, type
FROM sqlite_master
WHERE type IN ('table', 'view')
ORDER BY type, name
""", conn)
tables

## 3. Master KPI Snapshot

This single query powers executive summary cards in Excel and Streamlit.

In [ ]:
master_kpis_sql = (SQL_DIR / "kpis" / "master_kpis.sql").read_text(encoding="utf-8")
kpis = pd.read_sql_query(master_kpis_sql, conn)
kpis

## 4. Revenue Trend

Monthly revenue with month-over-month growth highlights seasonality and campaign impact.

In [ ]:
revenue_sql = """
WITH monthly AS (
    SELECT d.year, d.month, printf('%04d-%02d', d.year, d.month) AS year_month,
           ROUND(SUM(o.line_total), 2) AS revenue,
           COUNT(DISTINCT o.invoice_no) AS orders
    FROM fact_orders o
    JOIN dim_date d ON o.date_id = d.date_id
    GROUP BY d.year, d.month
)
SELECT year_month, revenue, orders,
       ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY year, month))
             / NULLIF(LAG(revenue) OVER (ORDER BY year, month), 0), 2) AS mom_growth_pct
FROM monthly
ORDER BY year, month
"""
monthly_revenue = pd.read_sql_query(revenue_sql, conn)
monthly_revenue

In [ ]:
try:
    import matplotlib.pyplot as plt
    ax = monthly_revenue.plot(x="year_month", y="revenue", kind="bar", figsize=(12, 4), legend=False, title="Monthly Revenue")
    ax.set_xlabel("Month")
    ax.set_ylabel("Revenue")
    plt.tight_layout()
except ImportError:
    print("matplotlib is not installed; showing the revenue table instead.")
    display(monthly_revenue)

## 5. Customer RFM Segmentation

RFM converts purchase behavior into actionable customer groups for lifecycle marketing.

In [ ]:
rfm_sql = (SQL_DIR / "analysis" / "02_customer_rfm.sql").read_text(encoding="utf-8").split(";")[0]
rfm = pd.read_sql_query(rfm_sql, conn)
rfm.head(10)

In [ ]:
rfm_summary = (
    rfm.groupby("rfm_segment", as_index=False)
       .agg(customers=("customer_id", "count"), revenue=("monetary", "sum"), avg_value=("monetary", "mean"))
       .sort_values("revenue", ascending=False)
)
rfm_summary

## 6. Product and Category Performance

Category-level analysis combines revenue, margin, and returns to identify both growth opportunities and quality risks.

In [ ]:
category_sql = (SQL_DIR / "analysis" / "03_product_performance.sql").read_text(encoding="utf-8").split(";")[1]
category_perf = pd.read_sql_query(category_sql, conn)
category_perf

## 7. Pricing Intelligence

The synthetic fallback data includes deliberate pre-sale price increases for a subset of products so the detection SQL has a realistic signal to find.

In [ ]:
dark_pattern_sql = (SQL_DIR / "analysis" / "05_dark_pattern_detection.sql").read_text(encoding="utf-8").split(";")[0]
dark_patterns = pd.read_sql_query(dark_pattern_sql, conn)
dark_patterns.head(20)

## 8. Return Risk and Fraud Flags

Return analysis surfaces unusually high return rates and high-value clusters by city tier and loyalty tier.

In [ ]:
returns_sql = (SQL_DIR / "analysis" / "06_return_fraud_analysis.sql").read_text(encoding="utf-8").split(";")[0]
return_flags = pd.read_sql_query(returns_sql, conn)
return_flags.head(20)

## 9. City Tier Insights

City-tier performance is useful for localized campaigns, inventory planning, and retention strategy.

In [ ]:
city_sql = (SQL_DIR / "analysis" / "07_city_tier_analysis.sql").read_text(encoding="utf-8").split(";")[0]
city_tier = pd.read_sql_query(city_sql, conn)
city_tier

## 10. Business Takeaways

Use the outputs above to write final project insights in the README and dashboard story. Lead with revenue growth, strongest category, RFM segment opportunities, pricing volatility, return risks, and data quality governance.

In [ ]:
conn.close()
print("Notebook analysis complete.")